# Class Activity: SQL, Byond Basic
 - Linking tables with Foreign Keys.
-  Combining data with INNER, LEFT, and OUTER JOINs.
- Summarizing data with GROUP BY, SUM, and AVG.
- Updating records and Altering Table Schema.


In [5]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = ON;")
cursor = conn.cursor()



## Practice Join

In [6]:
cursor.executescript('''
CREATE TABLE depts (dept_id INTEGER PRIMARY KEY, dept_name TEXT);
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, 
    name TEXT, 
    dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES depts(dept_id)
);
''')

cursor.execute("INSERT INTO depts VALUES (1, 'Engineering'), (2, 'Marketing'), (3, 'Finance')")

cursor.execute("INSERT INTO employees VALUES (101, 'Alice', 1), (102, 'Bob', 2), (103, 'Charlie', NULL)")

In [7]:
print("INNER JOIN: Matches Only\n")

sql ="SELECT e.name, d.dept_name FROM employees e JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("LEFT JOIN: All Employees\n") 
sql = "SELECT e.name, d.dept_name FROM employees e LEFT JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("RIGHT JOIN: All Departments\n")
sql = "SELECT e.name, d.dept_name FROM employees e RIGHT JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

print('#'*50)
print("FULL OUTER JOIN: The Whole Picture\n")
sql = "SELECT e.name, d.dept_name FROM employees e FULL OUTER JOIN depts d ON e.dept_id = d.dept_id"
print(pd.read_sql_query(sql, conn))

INNER JOIN: Matches Only

    name    dept_name
0  Alice  Engineering
1    Bob    Marketing
##################################################
LEFT JOIN: All Employees

      name    dept_name
0    Alice  Engineering
1      Bob    Marketing
2  Charlie         None
##################################################
RIGHT JOIN: All Departments

    name    dept_name
0  Alice  Engineering
1    Bob    Marketing
2   None      Finance
##################################################
FULL OUTER JOIN: The Whole Picture

      name    dept_name
0    Alice  Engineering
1      Bob    Marketing
2  Charlie         None
3     None      Finance


In [8]:
try:
    cursor.execute("INSERT INTO employees VALUES (11, 'John', 5)")
except sqlite3.IntegrityError as e:
    print(f"BLOCKED: {e} (Could be attempting to add an employee with no valid department)")

BLOCKED: FOREIGN KEY constraint failed (Could be attempting to add an employee with no valid department)


### Aggregate Functions (Summarizing Data)
SUM, COUNT, AVG, MIN, MAX


In [9]:
# Create Categories and Products Tables
cursor.execute("CREATE TABLE IF NOT EXISTS categories (id INTEGER PRIMARY KEY, cat_name TEXT)")
cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY, 
        name TEXT, 
        price REAL, 
        cat_id INTEGER,
        FOREIGN KEY (cat_id) REFERENCES categories(id)
    )""")

# Insert DATA
cursor.execute("INSERT OR REPLACE INTO categories VALUES (1, 'Electronics'), (2, 'Furniture')")
cursor.executemany("INSERT INTO products (name, price, cat_id) VALUES (?,?,?)", [
    ('Laptop', 1200.00, 1),
    ('Phone', 800.00, 1),
    ('Desk', 250.00, 2),
    ('Chair', 150.00, 2),
    ('Monitor', 300.00, 1)
])

In [10]:
print("BASIC GROUP BY (Count products per category)")

query1 = """
SELECT c.cat_name, COUNT(p.id) as total_products
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
"""
print(pd.read_sql_query(query1, conn))


print("\nFINANCIAL SUMMARY (Average & Total Price)")

query2 = """
SELECT c.cat_name, 
      # SUM(p.price) as inventory_value,
      # ROUND(AVG(p.price), 2) as average_price
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
"""
print(pd.read_sql_query(query2, conn))


BASIC GROUP BY (Count products per category)
      cat_name  total_products
0  Electronics               3
1    Furniture               2

FINANCIAL SUMMARY (Average & Total Price)
      cat_name  inventory_value  average_price
0  Electronics           2300.0         766.67
1    Furniture            400.0         200.00


 ### Advanced Analytics Using the HAVING Filter
 'WHERE' filters rows before grouping. 'HAVING' filters the results AFTER grouping.

In [11]:
query3 = """
SELECT c.cat_name, SUM(p.price) as total_value
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
HAVING total_value > 500
"""
print(pd.read_sql_query(query3, conn))

      cat_name  total_value
0  Electronics       2300.0


### Which category has the single most expensive item?

In [12]:
query4 = """
SELECT c.cat_name, MAX(p.price) as max_value
FROM categories c
JOIN products p ON c.id = p.cat_id
GROUP BY c.cat_name
ORDER BY max_value DESC
LIMIT 2
"""
print(pd.read_sql_query(query4, conn))

      cat_name  max_value
0  Electronics     1200.0
1    Furniture      250.0


### Extra Activity
Use
"ALTER TABLE table_name ADD COLUMN column_name data_type"
1. To add "quantity" and "discount" column to product tables.
2. update this values for each product
3. find the product with quantity less than 5 
4. Find the categories with maximum discount
5. What is the total inventory and total inventroy value?

In [26]:
#1. 
#conn.execute('ALTER TABLE products ADD COLUMN quantity')
#I got an error because I already added it, and it wouldn't let me add it twice.

In [29]:
#2. update this values for each product
conn.execute("UPDATE products SET quantity = 10, discount = 0.05 WHERE id = 1")
conn.execute("UPDATE products SET quantity = 2,  discount = 0.20 WHERE id = 2")
conn.execute("UPDATE products SET quantity = 15, discount = 0.10 WHERE id = 3")
conn.execute("UPDATE products SET quantity = 4,  discount = 0.15 WHERE id = 4")
conn.commit()

In [39]:
# 3. find the product with quantity less than 5
query_less_5 = "SELECT * FROM products WHERE quantity < 5"
pd.read_sql_query(query_less_5, conn)

,id,name,price,cat_id,quantity,discount
0,2,Phone,800.0,1,2,0.20
1,4,Chair,150.0,2,4,0.15


In [40]:
# 4. Find the categories with maximum discount
query_max_discount = """ 
SELECT * FROM products
""" 
pd.read_sql_query(query_max_discount, conn)

,id,name,price,cat_id,quantity,discount
0,1,Laptop,1200.0,1,10.0,0.05
1,2,Phone,800.0,1,2.0,0.20
2,3,Desk,250.0,2,15.0,0.10
3,4,Chair,150.0,2,4.0,0.15
4,5,Monitor,300.0,1,NaN,NaN


In [45]:
#5. What is the total inventory and total inventory value?
query_total = """
SELECT
    SUM(quantity) AS total_inventory,
    SUM(price*quantity) AS total_inventory_value
FROM products
"""
pd.read_sql_query(query_total, conn)

,total_inventory,total_inventory_value
0,31,17950.0
